In [1]:
import pandas as pd
import numpy as np
import random
import os
from datetime import datetime, timedelta

# ==========================================
# SETUP & CONFIGURATION
# ==========================================
CDR_FILE = "cdr_dataset.csv"
BANK_FILE = "final_bank_transactions_with_fraud.csv"
OUTPUT_FILE = "ipdr_dataset.csv"
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Randomly select a row count between 30,000 and 60,000
TOTAL_SESSIONS = random.randint(30000, 60000)
UNIQUE_USERS = int(TOTAL_SESSIONS * 0.15) # Assume average of ~6-7 sessions per user

CITIES = ["Mumbai", "Delhi", "Bengaluru", "Hyderabad", "Chennai", "Kolkata", "Pune", "Ahmedabad", "Jaipur", "Lucknow"]
ISPS = ["Airtel Broadband", "JioFiber", "BSNL", "Vodafone Idea", "ACT Fibernet"]
BROWSERS = ["Chrome", "Safari", "Edge", "Firefox", "App_WebView"]

# ==========================================
# HELPER FUNCTIONS
# ==========================================
def generate_ipv4(is_public=True, international=False):
    if not is_public:
        return f"192.168.{random.randint(0,255)}.{random.randint(1,254)}"
    if international:
        # Faking international ranges (e.g., US, Europe)
        return f"{random.choice([104, 185, 193, 82])}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}"
    # Indian IP ranges (approximate mock)
    return f"{random.choice([103, 122, 115, 49])}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}"

def load_or_mock_identities(num_needed):
    """Extracts entities from existing datasets to ensure overlap, or generates synthetics."""
    identities = []
    
    # Try CDR First (best match for MSISDN, IMEI, Subscriber)
    if os.path.exists(CDR_FILE):
        print(f"Loading overlap identities from {CDR_FILE}...")
        df = pd.read_csv(CDR_FILE)
        subset = df[['Caller_MSISDN', 'Subscriber_ID', 'Caller_Name', 'IMEI', 'Device_Model', 'Device_OS', 'Tower_City']].dropna().drop_duplicates(subset=['Caller_MSISDN'])
        
        for _, row in subset.head(num_needed).iterrows():
            identities.append({
                "MSISDN": str(row['Caller_MSISDN']).replace('.0', ''),
                "Subscriber_ID": str(row['Subscriber_ID']),
                "Name": str(row['Caller_Name']),
                "IMEI": str(row['IMEI']).replace('.0', ''),
                "Device_Model": str(row['Device_Model']),
                "OS": str(row['Device_OS']),
                "City": str(row['Tower_City'])
            })
            
    # Try Bank Second
    elif os.path.exists(BANK_FILE):
        print(f"Loading overlap identities from {BANK_FILE}...")
        df = pd.read_csv(BANK_FILE)
        subset = df[['Sender_Phone_Number', 'Sender_Customer_Name', 'Sender_IMEI', 'Sender_City']].dropna().drop_duplicates(subset=['Sender_Phone_Number'])
        
        for idx, row in subset.head(num_needed).iterrows():
            identities.append({
                "MSISDN": str(row['Sender_Phone_Number']).replace('.0', ''),
                "Subscriber_ID": f"SUB_{str(idx).zfill(6)}",
                "Name": str(row['Sender_Customer_Name']),
                "IMEI": str(row['Sender_IMEI']).replace('.0', ''),
                "Device_Model": random.choice(["Samsung Galaxy S24", "iPhone 15", "OnePlus 12", "Redmi Note 13"]),
                "OS": random.choice(["Android", "iOS"]),
                "City": str(row['Sender_City'])
            })

    # Fill remainder
    remaining = num_needed - len(identities)
    if remaining > 0:
        print(f"Generating {remaining} synthetic identities...")
        for i in range(remaining):
            identities.append({
                "MSISDN": str(random.randint(9000000000, 9999999999)),
                "Subscriber_ID": f"SUB_SYN_{str(i).zfill(5)}",
                "Name": f"User_{random.randint(1000,99999)}",
                "IMEI": str(random.randint(100000000000000, 999999999999999)),
                "Device_Model": random.choice(["Samsung Galaxy S24", "iPhone 15", "OnePlus 12", "Redmi Note 13"]),
                "OS": random.choice(["Android", "iOS", "Windows"]),
                "City": random.choice(CITIES)
            })
            
    return pd.DataFrame(identities).sample(frac=1).reset_index(drop=True)

def main():
    print(f"--- Starting IPDR Generation (Target: {TOTAL_SESSIONS} Sessions) ---")
    
    # 1. Base Identities (Strict 1-to-1 Mapping mappings)
    users_df = load_or_mock_identities(UNIQUE_USERS)
    
    # Create strict 1-to-1 mappings
    msisdn_to_sub = dict(zip(users_df['MSISDN'], users_df['Subscriber_ID']))
    msisdn_to_name = dict(zip(users_df['MSISDN'], users_df['Name']))
    msisdn_to_city = dict(zip(users_df['MSISDN'], users_df['City']))
    
    # Generate Device IDs mapping strictly to IMEIs
    unique_imeis = users_df['IMEI'].unique()
    imei_to_devid = {imei: f"DEV_{str(i+1).zfill(6)}" for i, imei in enumerate(unique_imeis)}
    users_df['Device_ID'] = users_df['IMEI'].map(imei_to_devid)
    
    # Assign Fraud Personas
    fraud_categories = ["Normal", "Account_Takeover", "Credential_Stuffing", "Bot_Activity", "Suspicious_Login", "Data_Exfiltration", "Mule_Network"]
    probs = [0.95, 0.01, 0.008, 0.008, 0.008, 0.008, 0.008]
    users_df['Fraud_Persona'] = np.random.choice(fraud_categories, p=probs, size=len(users_df))

    # 2. Generate Sessions
    sessions = []
    start_date = datetime(2023, 10, 1)
    
    # Convert users_df to dict for fast O(1) lookup during loop
    user_records = users_df.to_dict('records')
    
    for i in range(TOTAL_SESSIONS):
        u = random.choice(user_records)
        
        # Base Session variables
        session_start = start_date + timedelta(days=random.uniform(0, 30), hours=random.uniform(0, 24))
        duration = int(random.uniform(30, 7200)) # 30s to 2 hours
        session_end = session_start + timedelta(seconds=duration)
        
        net_type = random.choice(["4G", "5G", "WiFi", "Fiber"])
        conn_type = "Mobile_Data" if net_type in ["4G", "5G"] else random.choice(["Home_Broadband", "Public_WiFi"])
        
        up_mb = round(random.uniform(0.1, 50.0), 2)
        down_mb = round(random.uniform(1.0, 500.0), 2)
        
        is_international = False
        is_vpn = False
        is_suspicious = False
        is_data_spike = False
        pub_ip = generate_ipv4()
        location_city = u['City']
        
        fraud_type = u['Fraud_Persona']
        investigation = "Not_Investigated"
        
        # Apply Fraud Logic Mutations
        if fraud_type != "Normal":
            investigation = random.choice(["Under_Review", "Confirmed_Fraud", "Closed"])
            
            if fraud_type == "Account_Takeover":
                is_vpn = True
                is_suspicious = True
                pub_ip = generate_ipv4(international=random.choice([True, False]))
                is_international = pub_ip.startswith(('104', '185', '193', '82'))
                location_city = random.choice(CITIES) # Location mismatch
                
            elif fraud_type == "Data_Exfiltration":
                up_mb = round(random.uniform(1000.0, 5000.0), 2) # Massive upload
                is_data_spike = True
                
            elif fraud_type == "Credential_Stuffing":
                duration = int(random.uniform(1, 10)) # Micro sessions
                is_suspicious = True
                is_vpn = True
                
            elif fraud_type == "Bot_Activity":
                duration = int(random.uniform(3600, 14400))
                down_mb = round(random.uniform(10.0, 20.0), 2)
                up_mb = round(random.uniform(10.0, 20.0), 2)
                
            elif fraud_type == "Mule_Network":
                conn_type = "Public_WiFi"
                location_city = random.choice(CITIES)
                
        # Construct Record
        sessions.append({
            "IPDR_ID": f"IPDR_{str(i+1).zfill(6)}",
            "Session_ID": f"SESS_{str(random.randint(10000000, 99999999))}",
            "Subscriber_ID": msisdn_to_sub[u['MSISDN']],
            "User_MSISDN": u['MSISDN'],
            "User_Name": msisdn_to_name[u['MSISDN']],
            
            "Session_Start_Time": session_start.strftime("%Y-%m-%d %H:%M:%S"),
            "Session_End_Time": session_end.strftime("%Y-%m-%d %H:%M:%S"),
            "Session_Duration_Seconds": duration,
            "Data_Usage_MB": round(up_mb + down_mb, 2),
            "Upload_MB": up_mb,
            "Download_MB": down_mb,
            
            "Public_IP_Address": pub_ip,
            "Private_IP_Address": generate_ipv4(is_public=False),
            "ISP_Name": random.choice(ISPS),
            "Network_Type": net_type,
            "Connection_Type": conn_type,
            
            "Device_ID": u['Device_ID'],
            "IMEI": u['IMEI'],
            "Device_Model": u['Device_Model'],
            "Operating_System": u['OS'],
            "Browser": random.choice(BROWSERS),
            
            "IP_Location_City": location_city,
            "Latitude": round(random.uniform(8.0, 37.0), 6),
            "Longitude": round(random.uniform(68.0, 97.0), 6),
            "Location_Change_Count": random.randint(0, 5) if fraud_type != "Normal" else random.randint(0, 1),
            
            "Daily_Session_Count": random.randint(1, 20) if fraud_type != "Bot_Activity" else random.randint(50, 200),
            "Unique_IP_Count": random.randint(1, 3) if not is_vpn else random.randint(5, 15),
            "Login_Frequency": random.randint(1, 5) if fraud_type != "Credential_Stuffing" else random.randint(20, 100),
            
            "Suspicious_Login_Flag": 1 if is_suspicious else 0,
            "VPN_Proxy_Flag": 1 if is_vpn else 0,
            "International_IP_Flag": 1 if is_international else 0,
            "Data_Spike_Flag": 1 if is_data_spike else 0,
            
            "Fraud_Flag": 1 if fraud_type != "Normal" else 0,
            "Fraud_Type": fraud_type,
            "Investigation_Status": investigation
        })
        
    df = pd.DataFrame(sessions)
    
    # Sort chronologically for realism
    df = df.sort_values('Session_Start_Time').reset_index(drop=True)
    
    # Re-assign sequential IPDR_IDs after sorting
    df['IPDR_ID'] = [f"IPDR_{str(i+1).zfill(6)}" for i in range(len(df))]
    
    # Save file
    df.to_csv(OUTPUT_FILE, index=False)
    
    # ==========================================
    # DATA QUALITY & REPORT GENERATION
    # ==========================================
    print("\n" + "="*50)
    print("               IPDR DATASET REPORT")
    print("="*50)
    
    print(f"\n1. Shape: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"\n2. Column List:\n   {', '.join(df.columns.tolist())}")
    
    print("\n3. Data Types:")
    print(df.dtypes.to_string())
    
    print("\n4. First 5 Rows (Selected Columns):")
    cols_to_show = ['IPDR_ID', 'User_MSISDN', 'Session_Start_Time', 'Public_IP_Address', 'Data_Usage_MB', 'Fraud_Type']
    print(df[cols_to_show].head().to_string(index=False))
    
    print("\n5. Missing Value Report:")
    missing = df.isnull().sum()
    print("   No missing values!" if missing.sum() == 0 else missing[missing > 0].to_string())
    
    print("\n6. Duplicate Report:")
    dup_ipdr = df['IPDR_ID'].duplicated().sum()
    dup_rows = df.duplicated().sum()
    print(f"   Duplicate IPDR_IDs: {dup_ipdr}")
    print(f"   Duplicate Rows: {dup_rows}")
    
    print("\n7. Fraud Distribution:")
    print(df['Fraud_Type'].value_counts().to_string())
    print(f"\n   Total Fraud Flag Ratio: {(df['Fraud_Flag'].mean() * 100):.2f}% Fraud")
    
    print("\n8. Entity Consistency Checks:")
    msisdn_sub_check = df.groupby('User_MSISDN')['Subscriber_ID'].nunique().max()
    msisdn_name_check = df.groupby('User_MSISDN')['User_Name'].nunique().max()
    device_imei_check = df.groupby('Device_ID')['IMEI'].nunique().max()
    
    print(f"   User_MSISDN -> Subscriber_ID Max Unique: {msisdn_sub_check} (Expected: 1)")
    print(f"   User_MSISDN -> User_Name Max Unique:     {msisdn_name_check} (Expected: 1)")
    print(f"   Device_ID -> IMEI Max Unique:            {device_imei_check} (Expected: 1)")
    
    if all(v == 1 for v in [msisdn_sub_check, msisdn_name_check, device_imei_check]):
        print("   [✓] ALL ENTITY CONSISTENCY CHECKS PASSED.")
    else:
        print("   [✗] WARNING: CONSISTENCY RULE VIOLATED.")

if __name__ == "__main__":
    main()

--- Starting IPDR Generation (Target: 50952 Sessions) ---
Loading overlap identities from cdr_dataset.csv...

               IPDR DATASET REPORT

1. Shape: 50952 rows, 35 columns

2. Column List:
   IPDR_ID, Session_ID, Subscriber_ID, User_MSISDN, User_Name, Session_Start_Time, Session_End_Time, Session_Duration_Seconds, Data_Usage_MB, Upload_MB, Download_MB, Public_IP_Address, Private_IP_Address, ISP_Name, Network_Type, Connection_Type, Device_ID, IMEI, Device_Model, Operating_System, Browser, IP_Location_City, Latitude, Longitude, Location_Change_Count, Daily_Session_Count, Unique_IP_Count, Login_Frequency, Suspicious_Login_Flag, VPN_Proxy_Flag, International_IP_Flag, Data_Spike_Flag, Fraud_Flag, Fraud_Type, Investigation_Status

3. Data Types:
IPDR_ID                      object
Session_ID                   object
Subscriber_ID                object
User_MSISDN                  object
User_Name                    object
Session_Start_Time           object
Session_End_Time           

In [2]:
## validation code:

In [3]:
import pandas as pd
import numpy as np
import re


FILE = "ipdr_dataset.csv"

df = pd.read_csv(FILE)


# =====================================================
# BASIC INFORMATION
# =====================================================

print("="*80)
print("IPDR DATASET BASIC INFORMATION")
print("="*80)

print("\nShape:")
print(df.shape)


print("\n" + "="*80)
print("COLUMN NAMES")
print("="*80)

for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")


print("\n" + "="*80)
print("DATA TYPES")
print("="*80)

print(df.dtypes)



# =====================================================
# HEAD / TAIL
# =====================================================

print("\n" + "="*80)
print("FIRST 5 ROWS")
print("="*80)

print(df.head())


print("\n" + "="*80)
print("LAST 5 ROWS")
print("="*80)

print(df.tail())



# =====================================================
# MISSING VALUES
# =====================================================

print("\n" + "="*80)
print("MISSING VALUE CHECK")
print("="*80)

missing = df.isnull().sum()

if missing.sum() == 0:
    print("No missing values")
else:
    print(missing[missing > 0])



# =====================================================
# DUPLICATES
# =====================================================

print("\n" + "="*80)
print("DUPLICATE CHECK")
print("="*80)

print("Duplicate rows:", df.duplicated().sum())



# =====================================================
# UNIQUE COUNTS
# =====================================================

print("\n" + "="*80)
print("IMPORTANT UNIQUE COUNTS")
print("="*80)

important_cols = [
    "IPDR_ID",
    "Session_ID",
    "Subscriber_ID",
    "User_MSISDN",
    "Device_ID",
    "IMEI",
    "Public_IP_Address",
    "Fraud_Type"
]


for col in important_cols:
    if col in df.columns:
        print(f"{col}: {df[col].nunique()} unique values")



# =====================================================
# IMPORTANT COLUMN SAMPLES
# =====================================================

print("\n" + "="*80)
print("IMPORTANT COLUMN SAMPLES")
print("="*80)


for col in important_cols:
    if col in df.columns:
        print("\n", col)
        print(df[col].head(10).tolist())



# =====================================================
# PHONE FORMAT CHECK
# =====================================================

print("\n" + "="*80)
print("PHONE FORMAT CHECK")
print("="*80)


if "User_MSISDN" in df.columns:

    phone_length = (
        df["User_MSISDN"]
        .astype(str)
        .str.replace(".0","")
        .str.len()
        .value_counts()
    )

    print(phone_length)



# =====================================================
# IMEI CHECK
# =====================================================

print("\n" + "="*80)
print("IMEI LENGTH CHECK")
print("="*80)


if "IMEI" in df.columns:

    print(
        df["IMEI"]
        .astype(str)
        .str.replace(".0","")
        .str.len()
        .value_counts()
    )



# =====================================================
# IP ADDRESS VALIDATION
# =====================================================

print("\n" + "="*80)
print("IP ADDRESS CHECK")
print("="*80)


def valid_ip(ip):

    pattern = r"^(?:[0-9]{1,3}\.){3}[0-9]{1,3}$"

    return bool(re.match(pattern,str(ip)))


if "Public_IP_Address" in df.columns:

    invalid_ips = df[
        ~df["Public_IP_Address"].apply(valid_ip)
    ]

    print("Invalid IP count:", len(invalid_ips))



# =====================================================
# FRAUD DISTRIBUTION
# =====================================================

print("\n" + "="*80)
print("FRAUD DISTRIBUTION")
print("="*80)


if "Fraud_Flag" in df.columns:
    print(df["Fraud_Flag"].value_counts())


if "Fraud_Type" in df.columns:
    print("\nFraud Types:")
    print(df["Fraud_Type"].value_counts())



# =====================================================
# SESSION STATISTICS
# =====================================================

print("\n" + "="*80)
print("SESSION STATISTICS")
print("="*80)


numeric_cols = [
    "Session_Duration_Seconds",
    "Data_Usage_MB",
    "Upload_MB",
    "Download_MB",
    "Daily_Session_Count",
    "Unique_IP_Count"
]


for col in numeric_cols:
    if col in df.columns:
        print("\n",col)
        print(df[col].describe())



# =====================================================
# ENTITY CONSISTENCY CHECKS
# =====================================================

print("\n" + "="*80)
print("ENTITY CONSISTENCY CHECK")
print("="*80)


checks = [
    ("User_MSISDN","Subscriber_ID"),
    ("User_MSISDN","User_Name"),
    ("Device_ID","IMEI")
]


for col1,col2 in checks:

    if col1 in df.columns and col2 in df.columns:

        print(f"\nChecking {col1} --> {col2}")

        result = df.groupby(col1)[col2].nunique()

        bad = result[result > 1]

        if len(bad)==0:
            print("PASSED")
        else:
            print("FAILED")
            print("Inconsistent entities:",len(bad))
            print(bad.head())



print("\n" + "="*80)
print("IPDR CHECK COMPLETE")
print("="*80)

IPDR DATASET BASIC INFORMATION

Shape:
(50952, 35)

COLUMN NAMES
1. IPDR_ID
2. Session_ID
3. Subscriber_ID
4. User_MSISDN
5. User_Name
6. Session_Start_Time
7. Session_End_Time
8. Session_Duration_Seconds
9. Data_Usage_MB
10. Upload_MB
11. Download_MB
12. Public_IP_Address
13. Private_IP_Address
14. ISP_Name
15. Network_Type
16. Connection_Type
17. Device_ID
18. IMEI
19. Device_Model
20. Operating_System
21. Browser
22. IP_Location_City
23. Latitude
24. Longitude
25. Location_Change_Count
26. Daily_Session_Count
27. Unique_IP_Count
28. Login_Frequency
29. Suspicious_Login_Flag
30. VPN_Proxy_Flag
31. International_IP_Flag
32. Data_Spike_Flag
33. Fraud_Flag
34. Fraud_Type
35. Investigation_Status

DATA TYPES
IPDR_ID                      object
Session_ID                   object
Subscriber_ID                object
User_MSISDN                   int64
User_Name                    object
Session_Start_Time           object
Session_End_Time             object
Session_Duration_Seconds      in